# SML Opaque Entity Evaluation

**Tests whether an SML-trained model can reason from SML relational structure alone.**

100 multiple-choice questions using completely opaque entity tokens (X0, X1, ...) — the model has zero pretrained knowledge about these entities and MUST read the SML relations to answer correctly.

## Setup
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Set `HF_TOKEN` in Colab Secrets if using a gated model
3. Edit the **Configuration** cell with your model path
4. Run all cells in order


## 0. Install Dependencies

In [ ]:
%%capture
!pip install lm-eval>=0.4.4 transformers accelerate torch datasets pyyaml


## 1. Configuration

Set your model paths below. `MODEL_PATH` is the SML fine-tuned model to evaluate. `BASELINE_PATH` is the vanilla base model for comparison (set to `None` to skip).

In [ ]:
from pathlib import Path
import torch

# ── EDIT THESE ─────────────────────────────────────────────────────
MODEL_PATH    = 'sweetpapa/sml-qwen3-4b'   # Your SML fine-tuned model
BASELINE_PATH = 'Qwen/Qwen3-4B'            # Vanilla base model (or None)
DTYPE         = 'float16'                   # float16 or bfloat16
BATCH_SIZE    = 'auto'                      # 'auto' or integer
LIMIT         = None                        # Set to e.g. 10 for quick test
# ──────────────────────────────────────────────────────────────────

EVAL_DIR = Path('/content/sml_opaque_eval')
EVAL_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = EVAL_DIR / 'sml_opaque_reasoning.jsonl'
YAML_PATH  = EVAL_DIR / 'sml_opaque_reasoning.yaml'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f'GPU: {gpu_name}')
print(f'Device: {DEVICE}')
print(f'Model: {MODEL_PATH}')
print(f'Baseline: {BASELINE_PATH}')


## 2. Generate Evaluation Dataset (100 Questions)

Algorithmically generates 100 multiple-choice questions across 6 categories:
- **Simple Lookup** (20) — read a single relation
- **Negation** (15) — interpret NOT\_ relations
- **Multi-Hop** (20) — follow 2-3 relation chains
- **Weight Comparison** (15) — compare relation weights
- **Counting** (10) — count entities/relations
- **Composite** (20) — combine multiple reasoning skills

Includes 9 "no information" questions where the correct answer is that the SML data is insufficient.

In [ ]:
import json, random

LABELS = ['A', 'B', 'C', 'D']
LOOKUP_RELS = ['IsA', 'HasA', 'PartOf', 'AtLocation', 'CapableOf', 'UsedFor', 'HasProperty']
RELATION_QUESTION = {
    'IsA': 'what type of entity is {src}?', 'HasA': 'what does {src} have?',
    'PartOf': 'what is {src} a part of?', 'AtLocation': 'where is {src} located?',
    'CapableOf': 'what is {src} capable of?', 'UsedFor': 'what is {src} used for?',
    'HasProperty': 'what property does {src} have?', 'Causes': 'what does {src} cause?',
    'HasPrerequisite': 'what is a prerequisite for {src}?',
    'RelatedTo': 'what is {src} most related to?',
}
RELATION_NATURAL = {
    'RelatedTo': 'related', 'Causes': 'a cause of', 'SimilarTo': 'similar',
    'HasProperty': 'associated via property with', 'CapableOf': 'capable of something involving',
}

def E(idx, conf=0.9): return f'E(0|0|0|0|X{idx}|0|0|{conf})'
def R(rtype, src, tgt, weight=0.85): return f'R({rtype}|{src}|{tgt}|{weight:.2f}|0|0)'
def sml(ents, rels): return '<sml>\n' + '\n'.join(ents + rels) + '\n</sml>'

def fmt_question(sml_str, q_text, choices):
    labeled = [f'{LABELS[i]}) {choices[i]}' for i in range(len(choices))]
    return f'{sml_str}\n\n{q_text}\n' + '\n'.join(labeled)

def shuffle_choices(choices, correct_idx, rng):
    pairs = list(enumerate(choices)); rng.shuffle(pairs)
    new = [c for _, c in pairs]
    new_idx = next(i for i, (orig, _) in enumerate(pairs) if orig == correct_idx)
    return new, new_idx

def entry(sml_str, q_text, choices, correct_idx, rng, **meta):
    sh, idx = shuffle_choices(choices, correct_idx, rng)
    return {'question': fmt_question(sml_str, q_text, sh), 'choices': sh, 'answer': idx, **meta}

print('Question generator loaded.')


In [ ]:
# ═══ Category Generators ═══════════════════════════════════════════

def gen_simple_lookup(rng):
    questions = []
    for i in range(20):
        rel = LOOKUP_RELS[i % len(LOOKUP_RELS)]
        n = rng.randint(3, 6); conf = rng.choice([0.7, 0.8, 0.9, 1.0])
        w = round(rng.uniform(0.4, 1.0), 2); ents = [E(j, conf) for j in range(n)]
        if i >= 18:  # no-info
            tgt = rng.randint(1, n-1); ask_rel = rng.choice([r for r in LOOKUP_RELS if r != rel])
            rels = [R(rel, 0, tgt, w)]; s = sml(ents, rels)
            q = f'According to the SML data, {RELATION_QUESTION[ask_rel].format(src="X0")}'
            choices = [f'X{j}' for j in range(1, min(n,4))] + ['Not specified in the SML data']
            choices = choices[:4]
            while len(choices) < 4: choices.append('Cannot be determined')
            not_idx = next(k for k, c in enumerate(choices) if 'Not specified' in c)
            questions.append(entry(s, q, choices, not_idx, rng, category='simple_lookup', difficulty='easy', reasoning_type='no_information', num_entities=n, num_relations=1, num_hops=0))
            continue
        tgt = rng.randint(1, n-1); rels = [R(rel, 0, tgt, w)]; s = sml(ents, rels)
        q = f'According to the SML data, {RELATION_QUESTION[rel].format(src="X0")}'
        others = [j for j in range(1, n) if j != tgt]; dists = rng.sample(others, min(3, len(others)))
        choices = [f'X{tgt}'] + [f'X{d}' for d in dists]
        fillers = ['None of the above', 'Not specified', 'Cannot be determined']; fi = 0
        while len(choices) < 4 and fi < len(fillers): choices.append(fillers[fi]); fi += 1
        choices = choices[:4]
        questions.append(entry(s, q, choices, 0, rng, category='simple_lookup', difficulty='easy', reasoning_type='single_relation', num_entities=n, num_relations=1, num_hops=1))
    return questions

def gen_negation(rng):
    questions = []; neg_pairs = [('CapableOf','NOT_CapableOf'), ('HasProperty','NOT_HasProperty')]
    for i in range(15):
        pos_rel, neg_rel = neg_pairs[i % 2]; n = rng.randint(3, 5); conf = rng.choice([0.8, 0.9, 1.0])
        ents = [E(j, conf) for j in range(n)]
        if i == 14:  # no-info
            w = round(rng.uniform(0.6, 1.0), 2); rels = [R(pos_rel, 0, 1, w)]
            if n > 2: rels.append(R(pos_rel, 0, 2, round(rng.uniform(0.5, 0.9), 2)))
            s = sml(ents, rels)
            qw = 'capable of' if pos_rel == 'CapableOf' else 'a property of'
            q = f'According to the SML data, what is X0 NOT {qw}?'
            choices = [f'X{j}' for j in range(1, min(n,4))] + ['No negation is specified in the SML data']
            choices = choices[:4]
            while len(choices) < 4: choices.append('Cannot be determined')
            not_idx = next(k for k,c in enumerate(choices) if 'No negation' in c)
            questions.append(entry(s, q, choices, not_idx, rng, category='negation', difficulty='medium', reasoning_type='no_information', num_entities=n, num_relations=len(rels), num_hops=1))
            continue
        w_pos = round(rng.uniform(0.7, 1.0), 2); w_neg = round(rng.uniform(0.7, 1.0), 2)
        pos_tgt, neg_tgt = 1, 2
        if n < 3: n = 3; ents = [E(j, conf) for j in range(n)]
        rels = [R(pos_rel, 0, pos_tgt, w_pos), R(neg_rel, 0, neg_tgt, w_neg)]; s = sml(ents, rels)
        if i % 3 == 0:
            qw = 'capable of' if neg_rel == 'NOT_CapableOf' else 'a property of'
            q = f'Based on the SML data, what is X0 NOT {qw}?'
            choices = [f'X{neg_tgt}', f'X{pos_tgt}', f'Both X{pos_tgt} and X{neg_tgt}', 'Neither']
            questions.append(entry(s, q, choices, 0, rng, category='negation', difficulty='medium', reasoning_type='negation_lookup', num_entities=n, num_relations=2, num_hops=1))
        elif i % 3 == 1:
            qw = 'capable of' if pos_rel == 'CapableOf' else 'a property that X0 has'
            q = f'Based on the SML data, what is X0 {qw}?'
            choices = [f'X{pos_tgt}', f'X{neg_tgt}', f'Both X{pos_tgt} and X{neg_tgt}', 'Neither']
            questions.append(entry(s, q, choices, 0, rng, category='negation', difficulty='medium', reasoning_type='positive_with_negation_distractor', num_entities=n, num_relations=2, num_hops=1))
        else:
            cap = 'can do' if pos_rel == 'CapableOf' else 'has property'
            not_cap = 'cannot do' if neg_rel == 'NOT_CapableOf' else 'does not have property'
            choices = [f'X0 {cap} X{pos_tgt} but {not_cap} X{neg_tgt}', f'X0 {cap} X{pos_tgt} and X{neg_tgt}', f'X0 {not_cap} X{pos_tgt} or X{neg_tgt}', f'X0 {not_cap} X{pos_tgt} but {cap} X{neg_tgt}']
            q = 'Based on the SML data, which statement about X0 is true?'
            questions.append(entry(s, q, choices, 0, rng, category='negation', difficulty='medium', reasoning_type='negation_statement', num_entities=n, num_relations=2, num_hops=1))
    return questions

def gen_multi_hop(rng):
    questions = []
    chain_defs = [
        ('IsA','CapableOf','Based on the SML relations, which of the following can X0 do?','inheritance'),
        ('IsA','HasProperty','Based on the SML relations, which property can be inferred for X0?','inheritance'),
        ('AtLocation','PartOf','Based on the SML relations, within which larger entity is X0?','transitivity'),
        ('HasPrerequisite','HasPrerequisite','Based on the SML relations, what does X0 indirectly require?','prerequisite_chain'),
        ('Causes','Causes','Based on the SML relations, what does X0 indirectly cause?','causal_chain'),
    ]
    schedule = [0]*3 + [1]*2 + [2]*4 + [3]*4 + [4]*5; rng.shuffle(schedule)
    for i in range(20):
        if i >= 18:  # no-info
            n = rng.randint(4, 6); conf = rng.choice([0.8, 0.9]); ents = [E(j, conf) for j in range(n)]
            rels = [R('IsA', 0, 1, round(rng.uniform(0.6, 0.95), 2)), R('CapableOf', 1, 2, round(rng.uniform(0.6, 0.95), 2))]
            s = sml(ents, rels); d = 3 if n > 3 else 2
            q = f'Based on the SML relations, what can X{d} do?'
            others = [j for j in range(n) if j != d]; rng.shuffle(others)
            ce = list(dict.fromkeys([f'X{others[k]}' for k in range(min(3, len(others)))]))[:3]
            choices = ce + ['Insufficient information']
            while len(choices) < 4: choices.append('Cannot be determined')
            choices = choices[:4]; ins_idx = choices.index('Insufficient information')
            questions.append(entry(s, q, choices, ins_idx, rng, category='multi_hop', difficulty='hard', reasoning_type='no_information', num_entities=n, num_relations=2, num_hops=0))
            continue
        ci = schedule[i]; rel1, rel2, q_template, rtype = chain_defs[ci]
        hops = 3 if (i % 5 == 4 and ci in (3, 4)) else 2
        n_chain = hops + 1; nd = rng.randint(1, 3); n = n_chain + nd; conf = rng.choice([0.8, 0.9, 1.0])
        ents = [E(j, conf) for j in range(n)]; rels = []
        for h in range(hops):
            r = rel1 if h == 0 else rel2; rels.append(R(r, h, h+1, round(rng.uniform(0.6, 0.95), 2)))
        s = sml(ents, rels); end = n_chain - 1; correct = f'X{end}'
        others = [f'X{j}' for j in range(n) if j != end and j != 0]; dists = rng.sample(others, min(3, len(others)))
        choices = [correct] + dists
        while len(choices) < 4: choices.append('Insufficient information')
        choices = choices[:4]; diff = 'medium' if hops == 2 else 'hard'
        questions.append(entry(s, q_template, choices, 0, rng, category='multi_hop', difficulty=diff, reasoning_type=rtype, num_entities=n, num_relations=hops, num_hops=hops))
    return questions

def gen_weight_comparison(rng):
    questions = []; crels = ['RelatedTo','Causes','SimilarTo','HasProperty','CapableOf']
    for i in range(15):
        rel = crels[i % len(crels)]; n = rng.randint(3, 5); conf = rng.choice([0.8, 0.9, 1.0])
        ents = [E(j, conf) for j in range(n)]; nat = RELATION_NATURAL.get(rel, rel.lower())
        if i == 14:  # no-info
            w = round(rng.uniform(0.5, 0.95), 2); rels = [R(rel, 0, 1, w)]; s = sml(ents, rels)
            q = f'According to the SML data, is X0 more strongly {nat} to X1 or X2?'
            choices = [f'X1 (weight {w})', 'X2', 'They are equally related', 'Cannot compare — only one relation exists']
            questions.append(entry(s, q, choices, 3, rng, category='weight_comparison', difficulty='medium', reasoning_type='no_information', num_entities=n, num_relations=1, num_hops=1))
            continue
        if i in (11, 12):  # equal
            w = round(rng.uniform(0.4, 0.9), 2); rels = [R(rel, 0, 1, w), R(rel, 0, 2, w)]; s = sml(ents, rels)
            q = f'According to the SML data, is X0 more strongly {nat} to X1 or X2?'
            choices = ['X1', 'X2', f'They are equally related (both weight {w})', 'X0 is not related to either']
            questions.append(entry(s, q, choices, 2, rng, category='weight_comparison', difficulty='medium', reasoning_type='weight_equal', num_entities=n, num_relations=2, num_hops=1))
            continue
        if i < 5: w1=round(rng.uniform(0.75,0.98),2); w2=round(rng.uniform(0.10,0.35),2); diff='easy'
        elif i < 10: w1=round(rng.uniform(0.65,0.90),2); w2=round(rng.uniform(0.30,0.55),2); diff='medium'
        else: base=round(rng.uniform(0.50,0.80),2); w1=round(base+rng.uniform(0.05,0.12),2); w2=base; diff='hard'
        if rng.random() < 0.5: ht, lt = 1, 2
        else: ht, lt = 2, 1
        rels = [R(rel, 0, ht, w1), R(rel, 0, lt, w2)]; s = sml(ents, rels)
        q = f'According to the SML data, is X0 more strongly {nat} to X1 or X2?'
        choices = [f'X{ht} (weight {w1} vs {w2})', f'X{lt} (weight {w2} vs {w1})', 'They are equally related', 'X0 is not related to either']
        questions.append(entry(s, q, choices, 0, rng, category='weight_comparison', difficulty=diff, reasoning_type='weight_comparison', num_entities=n, num_relations=2, num_hops=1))
    return questions

def gen_counting(rng):
    questions = []
    for i in range(10):
        n = rng.randint(3, 7); conf = rng.choice([0.8, 0.9, 1.0]); ents = [E(j, conf) for j in range(n)]
        n_rels = rng.randint(2, min(6, n*(n-1)//2)); rts = ['IsA','HasA','CapableOf','AtLocation','Causes','PartOf']
        rels = []; used = set()
        for _ in range(n_rels):
            src = rng.randint(0, n-1); tgt = rng.randint(0, n-1)
            while tgt == src or (src, tgt) in used: src = rng.randint(0, n-1); tgt = rng.randint(0, n-1)
            used.add((src, tgt)); rels.append(R(rng.choice(rts), src, tgt, round(rng.uniform(0.3, 1.0), 2)))
        s = sml(ents, rels)
        if i == 9:  # no-info
            involved = set()
            for r in rels: p = r.split('|'); involved.add(int(p[1])); involved.add(int(p[2]))
            isolated = [j for j in range(n) if j not in involved]
            if not isolated: n += 1; ents.append(E(n-1, conf)); s = sml(ents, rels); isolated = [n-1]
            ae = rng.choice(isolated); q = f'How many relations in the SML block involve X{ae}?'
            choices = ['0', '1', '2', '3']
            questions.append(entry(s, q, choices, 0, rng, category='counting', difficulty='easy', reasoning_type='no_information', num_entities=n, num_relations=len(rels), num_hops=0))
            continue
        if i < 2:
            q = 'How many entities are defined in the SML block?'; correct = n
            choices = sorted(list(set([str(correct)] + [str(max(1,correct+d)) for d in [-1,1,2]])))[:4]
            while len(choices) < 4: choices.append(str(correct+3))
            questions.append(entry(s, q, choices, choices.index(str(correct)), rng, category='counting', difficulty='easy', reasoning_type='count_entities', num_entities=n, num_relations=len(rels), num_hops=0))
        elif i < 4:
            q = 'How many relations are defined in the SML block?'; correct = len(rels)
            choices = sorted(list(set([str(correct)] + [str(max(0,correct+d)) for d in [-1,1,-2]])))[:4]
            while len(choices) < 4: choices.append(str(correct+2))
            questions.append(entry(s, q, choices, choices.index(str(correct)), rng, category='counting', difficulty='easy', reasoning_type='count_relations', num_entities=n, num_relations=len(rels), num_hops=0))
        elif i < 6:
            outgoing = sum(1 for r in rels if r.split('|')[1] == '0')
            q = 'How many relations in the SML block have X0 as a source?'; correct = outgoing
            choices = sorted(list(set([str(correct)] + [str(max(0,correct+d)) for d in [-1,1,2]])))[:4]
            while len(choices) < 4: choices.append(str(correct+3))
            questions.append(entry(s, q, choices, choices.index(str(correct)), rng, category='counting', difficulty='medium', reasoning_type='count_outgoing', num_entities=n, num_relations=len(rels), num_hops=0))
        elif i < 8:
            rtset = set(r.split('(')[1].split('|')[0] for r in rels); correct = len(rtset)
            q = 'How many unique relation types appear in the SML block?'
            choices = sorted(list(set([str(correct)] + [str(max(1,correct+d)) for d in [-1,1,2]])))[:4]
            while len(choices) < 4: choices.append(str(correct+3))
            questions.append(entry(s, q, choices, choices.index(str(correct)), rng, category='counting', difficulty='medium', reasoning_type='count_unique_types', num_entities=n, num_relations=len(rels), num_hops=0))
        else:
            counts = {j: 0 for j in range(n)}
            for r in rels: p = r.split('|'); counts[int(p[1])] += 1; counts[int(p[2])] += 1
            most = max(counts, key=lambda k: counts[k])
            q = 'Which entity in the SML block has the most connections (as source or target)?'
            others = [j for j in range(n) if j != most]; dists = rng.sample(others, min(3, len(others)))
            choices = [f'X{most}'] + [f'X{d}' for d in dists]
            while len(choices) < 4: choices.append('All entities have equal connections')
            choices = choices[:4]
            questions.append(entry(s, q, choices, 0, rng, category='counting', difficulty='medium', reasoning_type='most_connected', num_entities=n, num_relations=len(rels), num_hops=0))
    return questions

def gen_composite(rng):
    questions = []
    for i in range(20):
        conf = rng.choice([0.8, 0.9, 1.0])
        if i >= 18:  # no-info
            n = rng.randint(4, 6); ents = [E(j, conf) for j in range(n)]
            rels = [R('IsA', 0, 1, round(rng.uniform(0.6, 0.95), 2)), R('HasProperty', 1, 2, round(rng.uniform(0.6, 0.95), 2))]
            s = sml(ents, rels); q = f'The SML block shows relationships between several entities. Based on the SML data, what is X{n-1} capable of?'
            choices = ['X0', 'X1', 'X2', 'Insufficient information']
            questions.append(entry(s, q, choices, 3, rng, category='composite', difficulty='hard', reasoning_type='no_information', num_entities=n, num_relations=2, num_hops=0))
            continue
        if i < 4:  # inheritance + negation
            n = rng.randint(4, 5); ents = [E(j, conf) for j in range(n)]
            rels = [R('IsA', 0, 1, round(rng.uniform(0.8, 0.95), 2)), R('CapableOf', 1, 2, round(rng.uniform(0.7, 0.9), 2)), R('NOT_CapableOf', 0, 2, round(rng.uniform(0.85, 0.99), 2))]
            s = sml(ents, rels)
            q = 'X0 is a type of X1, and X1 can do X2. However, X0 specifically cannot do X2. Based on the SML data, what does this tell us?'
            choices = ['X0 is an exception — it is a type of X1 that lacks the X2 capability', 'The SML data is contradictory and no conclusion can be drawn', 'X0 can do X2 because it inherits from X1', 'X1 also cannot do X2']
            questions.append(entry(s, q, choices, 0, rng, category='composite', difficulty='hard', reasoning_type='inheritance_negation', num_entities=n, num_relations=3, num_hops=2))
        elif i < 8:  # transitivity + comparison
            n = 5; ents = [E(j, conf) for j in range(n)]
            rels = [R('AtLocation', 0, 1, round(rng.uniform(0.6, 0.9), 2)), R('AtLocation', 1, 2, round(rng.uniform(0.6, 0.9), 2)), R('AtLocation', 3, 2, round(rng.uniform(0.6, 0.9), 2))]
            s = sml(ents, rels); q = 'X0 is at X1, X1 is at X2, and X3 is also at X2. Based on the SML data, are X0 and X3 in the same general area?'
            choices = ['Yes — both X0 and X3 are within X2', 'No — X0 is at X1, which is different from X3 location', 'Only if X1 and X3 are the same entity', 'Insufficient information to determine']
            questions.append(entry(s, q, choices, 0, rng, category='composite', difficulty='hard', reasoning_type='transitivity_comparison', num_entities=n, num_relations=3, num_hops=2))
        elif i < 12:  # chain + weight
            n = 4; ents = [E(j, conf) for j in range(n)]; wh = round(rng.uniform(0.80, 0.98), 2); wl = round(rng.uniform(0.10, 0.35), 2)
            rels = [R('Causes', 0, 1, wh), R('Causes', 0, 2, wl), R('Causes', 1, 3, round(rng.uniform(0.6, 0.9), 2))]; s = sml(ents, rels)
            q = f'X0 causes X1 (weight {wh}) and X0 causes X2 (weight {wl}). X1 also causes X3. What is X0 most likely to cause?'
            choices = [f'X1 (strongest direct cause at {wh})', f'X2 (weight {wl})', 'X3 (via X1)', 'All are equally likely']
            questions.append(entry(s, q, choices, 0, rng, category='composite', difficulty='hard', reasoning_type='chain_weight', num_entities=n, num_relations=3, num_hops=2))
        elif i < 15:  # chain + negation + contradiction
            n = 4; ents = [E(j, conf) for j in range(n)]
            rels = [R('HasPrerequisite', 0, 1, round(rng.uniform(0.7, 0.95), 2)), R('HasPrerequisite', 1, 2, round(rng.uniform(0.7, 0.95), 2)), R('NOT_CapableOf', 0, 2, round(rng.uniform(0.8, 0.99), 2))]
            s = sml(ents, rels); q = 'X0 requires X1, and X1 requires X2. But X0 is NOT capable of X2. Based on the SML data, what does this imply?'
            choices = ['X0 cannot satisfy its own prerequisites — it needs X2 but cannot do X2', 'X0 can still satisfy its prerequisites through X1', 'X1 is also not capable of X2', 'The prerequisites are optional']
            questions.append(entry(s, q, choices, 0, rng, category='composite', difficulty='hard', reasoning_type='chain_negation_contradiction', num_entities=n, num_relations=3, num_hops=2))
        else:  # weight + counting
            n = rng.randint(4, 6); ents = [E(j, conf) for j in range(n)]
            nr = rng.randint(3, min(5, n-1)); targets = rng.sample(range(1, n), nr)
            weights = [round(rng.uniform(0.2, 0.98), 2) for _ in range(nr)]
            rels = [R('RelatedTo', 0, t, w) for t, w in zip(targets, weights)]; s = sml(ents, rels)
            mw = max(weights); mt = targets[weights.index(mw)]; ah = sum(1 for w in weights if w > 0.5)
            q = f'X0 has {nr} relations to other entities with varying weights. Which entity is X0 most strongly related to, and how many relations have weight above 0.5?'
            correct_str = f'X{mt} is strongest; {ah} relations above 0.5'
            wt = next((t for t in targets if t != mt), targets[-1])
            distractors = [d for d in [f'X{wt} is strongest; {nr} relations above 0.5', f'X{mt} is strongest; {max(0,ah-1)} relations above 0.5', 'All relations are equally weighted'] if d != correct_str]
            fbs = [f'X{wt} is strongest; {ah} relations above 0.5', f'X{mt} is strongest; {ah+1} relations above 0.5']
            for fb in fbs:
                if len(distractors) >= 3: break
                if fb != correct_str and fb not in distractors: distractors.append(fb)
            choices = [correct_str] + distractors[:3]
            questions.append(entry(s, q, choices, 0, rng, category='composite', difficulty='hard', reasoning_type='weight_counting', num_entities=n, num_relations=nr, num_hops=1))
    return questions

print('All category generators loaded.')


In [ ]:
# ── Generate all 100 questions ───────────────────────────────────
rng = random.Random(42)
all_qs = []
all_qs.extend(gen_simple_lookup(rng));  print(f'  simple_lookup: {len(all_qs)}')
all_qs.extend(gen_negation(rng));       print(f'  + negation:    {len(all_qs)}')
all_qs.extend(gen_multi_hop(rng));      print(f'  + multi_hop:   {len(all_qs)}')
all_qs.extend(gen_weight_comparison(rng)); print(f'  + weight_cmp:  {len(all_qs)}')
all_qs.extend(gen_counting(rng));       print(f'  + counting:    {len(all_qs)}')
all_qs.extend(gen_composite(rng));      print(f'  + composite:   {len(all_qs)}')

assert len(all_qs) == 100, f'Expected 100, got {len(all_qs)}'

# Write JSONL
with open(JSONL_PATH, 'w') as f:
    for q in all_qs:
        f.write(json.dumps(q) + '\n')

# Summary
cats = {}
for q in all_qs: cats[q['category']] = cats.get(q['category'], 0) + 1
no_info = sum(1 for q in all_qs if q.get('reasoning_type') == 'no_information')
print(f'\nWrote {len(all_qs)} questions to {JSONL_PATH}')
print(f'Categories: {json.dumps(cats)}')
print(f'No-information questions: {no_info}')


## 3. Write lm-eval Task Config

In [ ]:
# ── Write YAML task config ───────────────────────────────────────
yaml_content = '''task: sml_opaque_reasoning
dataset_path: json
dataset_kwargs:
  data_files:
    test: sml_opaque_reasoning.jsonl
test_split: test
output_type: multiple_choice
doc_to_text: "You are an AI assistant that uses Structured Markup Language (SML) context to ground your reasoning. Always analyze the provided SML block before answering.\\n\\n{{question}}\\nAnswer:"
doc_to_choice: ["A", "B", "C", "D"]
doc_to_target: "{{answer}}"
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0
'''

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print(f'Task config written to {YAML_PATH}')

## 4. Run Evaluation — Fine-Tuned Model

Runs the SML opaque reasoning task against your fine-tuned model using lm-eval's log-likelihood multiple-choice scoring. The system prompt is included via `system_instruction` in the task config.

In [ ]:
import subprocess, sys, os

def run_eval(model_path, label='model'):
    """Run lm-eval and return exit code."""
    model_args = f'pretrained={model_path},dtype={DTYPE},trust_remote_code=True'
    cmd = [
        sys.executable, '-m', 'lm_eval',
        '--model', 'hf',
        '--model_args', model_args,
        '--tasks', 'sml_opaque_reasoning',
        '--include_path', str(EVAL_DIR),
        '--device', DEVICE,
        '--batch_size', str(BATCH_SIZE),
    ]
    if LIMIT is not None:
        cmd.extend(['--limit', str(LIMIT)])

    print(f'\n{"="*70}')
    print(f'Evaluating [{label}]: {model_path}')
    print(f'{"="*70}\n')
    print(f'Command: {" ".join(cmd)}\n')

    # Capture all output so it reliably appears in Colab cell output
    result = subprocess.run(
        cmd, cwd=str(EVAL_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    # Print everything lm-eval produced
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f'\nERROR: lm_eval exited with code {result.returncode}')
    return result.returncode

# Run fine-tuned model
run_eval(MODEL_PATH, label='SML Fine-Tuned')

## 5. (Optional) Run Baseline Model

Run the vanilla base model for comparison. A model with no SML training should score ~25% (random chance on 4-choice questions).

In [ ]:
if BASELINE_PATH:
    run_eval(BASELINE_PATH, label='Baseline')
else:
    print('BASELINE_PATH is None — skipping baseline. Set it in the config cell to enable.')


## 6. Results Summary & Interpretation

In [ ]:
# ── Category Breakdown ─────────────────────────────────────────
import json

with open(JSONL_PATH) as f:
    qs = [json.loads(line) for line in f if line.strip()]

cats = {}
diffs = {}
for q in qs:
    cats[q['category']] = cats.get(q['category'], 0) + 1
    diffs[q['difficulty']] = diffs.get(q['difficulty'], 0) + 1

print(f'{"Category":<22} {"Count":>6} {"Random":>8}')
print('-' * 40)
for cat, count in sorted(cats.items()):
    print(f'  {cat:<20} {count:>4}     ~25%')
print(f'  {"TOTAL":<20} {len(qs):>4}     ~25%')
print(f'\nDifficulty: {json.dumps(diffs)}')

print(f'''
{'='*50}
INTERPRETATION GUIDE
{'='*50}
  ~25%     = Random chance (no SML understanding)
  25-35%   = Marginal, possibly pattern-matching
  35-50%   = Some SML reasoning ability
  50-70%   = Significant SML reasoning (approach works!)
  >70%     = Strong SML reasoning augmentation
''')


## 7. Sample Questions (Sanity Check)

In [ ]:
# Show one question from each category
seen = set()
for q in qs:
    cat = q['category']
    if cat not in seen:
        seen.add(cat)
        ans = q['choices'][q['answer']]
        print(f'--- {cat} | answer={ans[:50]} ---')
        # Show just the question part (after SML block)
        parts = q['question'].split('</sml>')
        if len(parts) > 1:
            print(parts[1].strip()[:200])
        print()
